## Introdução 

Esta lição irá abordar: 
- O que é a chamada de função e os seus casos de uso 
- Como criar uma chamada de função usando o Azure OpenAI 
- Como integrar uma chamada de função numa aplicação 

## Objetivos de Aprendizagem 

Após completar esta lição, irá saber como e compreender: 

- O propósito de usar chamadas de função 
- Configurar Chamada de Função usando o Serviço Azure Open AI 
- Projetar chamadas de função eficazes para o caso de uso da sua aplicação 


## Compreender Chamadas de Funções 

Para esta lição, queremos construir uma funcionalidade para a nossa startup educativa que permita aos utilizadores usar um chatbot para encontrar cursos técnicos. Recomendaremos cursos que correspondam ao seu nível de competência, papel atual e tecnologia de interesse. 

Para completar isto, iremos usar uma combinação de: 
 - `Azure Open AI` para criar uma experiência de chat para o utilizador
 - `Microsoft Learn Catalog API` para ajudar os utilizadores a encontrar cursos com base no pedido do utilizador 
 - `Function Calling` para pegar na consulta do utilizador e enviá-la para uma função que fará o pedido à API. 

Para começar, vejamos por que motivo quereríamos usar a chamada de funções desde o início: 


### Porquê a Chamada de Funções 

Se já concluíste qualquer outra lição deste curso, provavelmente percebes o poder de usar Grandes Modelos de Linguagem (LLMs). Esperamos que também consigas ver algumas das suas limitações. 

A Chamada de Funções é uma funcionalidade do Serviço Azure Open AI para ultrapassar as seguintes limitações: 
1) Formato de resposta consistente 
2) Capacidade de usar dados de outras fontes de uma aplicação num contexto de chat 

Antes da chamada de funções, as respostas de um LLM eram não estruturadas e inconsistentes. Os programadores eram obrigados a escrever código de validação complexo para garantir que podiam tratar cada variação de resposta. 

Os utilizadores não conseguiam obter respostas como "Qual é o tempo atual em Estocolmo?". Isto porque os modelos estavam limitados ao momento em que os dados foram treinados. 

Vamos ver o exemplo abaixo que ilustra este problema: 

Suponhamos que queremos criar uma base de dados de dados de estudantes para podermos sugerir o curso certo a eles. Abaixo temos duas descrições de estudantes que são muito semelhantes nos dados que contêm.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Queremos enviar isto para um LLM para analisar os dados. Isto pode ser utilizado posteriormente na nossa aplicação para enviar isto para uma API ou armazenar numa base de dados. 

Vamos criar dois prompts idênticos nos quais instruímos o LLM sobre qual a informação que nos interessa: 


Queremos enviar isto para um LLM para analisar as partes que são importantes para o nosso produto. Assim, podemos criar dois prompts idênticos para instruir o LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Após criar estes dois prompts, iremos enviá-los para o LLM usando `client.responses.create`. Guardamos o prompt na variável `input` e atribuímos o papel a `user`. Isto é para imitar uma mensagem escrita por um utilizador a um chatbot. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Agora podemos enviar ambos os pedidos para o LLM e examinar a resposta que recebemos. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Mesmo que os prompts sejam os mesmos e as descrições sejam semelhantes, podemos obter formatos diferentes para a propriedade `Grades`. 

Se executar a célula acima várias vezes, o formato pode ser `3.7` ou `3.7 GPA`. 

Isto acontece porque o LLM recebe dados não estruturados na forma do prompt escrito e também devolve dados não estruturados. Precisamos de ter um formato estruturado para sabermos o que esperar ao armazenar ou utilizar estes dados.

Ao usar chamadas funcionais, podemos garantir que recebemos dados estruturados de volta. Ao usar chamadas de função, o LLM na verdade não chama nem executa nenhuma função. Em vez disso, criamos uma estrutura para o LLM seguir nas suas respostas. Depois usamos essas respostas estruturadas para saber que função executar nas nossas aplicações.  
 


![Diagrama do Fluxo de Chamada de Função](../../../../translated_images/pt-PT/Function-Flow.083875364af4f4bb.webp)


Podemos então pegar no que é retornado pela função e enviar isso de volta para o LLM. O LLM responderá então usando linguagem natural para responder à consulta do utilizador. 


### Casos de uso para a utilização de chamadas de função 

**Chamar Ferramentas Externas** 
Os chatbots são ótimos para fornecer respostas a perguntas dos utilizadores. Ao utilizar chamadas de função, os chatbots podem usar mensagens dos utilizadores para completar certas tarefas. Por exemplo, um estudante pode pedir ao chatbot para "Enviar um email ao meu professor a dizer que preciso de mais ajuda com esta matéria". Isto pode fazer uma chamada de função para `send_email(to: string, body: string)`


**Criar Consultas API ou de Base de Dados**
Os utilizadores podem encontrar informação usando linguagem natural que é convertida numa consulta formatada ou pedido API. Um exemplo disto pode ser um professor que pede "Quem são os estudantes que completaram a última tarefa" o que poderia chamar uma função chamada `get_completed(student_name: string, assignment: int, current_status: string)`


**Criar Dados Estruturados**
Os utilizadores podem pegar num bloco de texto ou CSV e usar o LLM para extrair informação importante dele. Por exemplo, um estudante pode converter um artigo da Wikipédia sobre acordos de paz para criar flashcards de IA. Isto pode ser feito usando uma função chamada `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Criar a Sua Primeira Chamada de Função 

O processo de criar uma chamada de função inclui 3 passos principais: 
1. Chamar a API de Chat Completions com uma lista das suas funções e uma mensagem do utilizador 
2. Ler a resposta do modelo para realizar uma ação, ou seja, executar uma função ou Chamada de API 
3. Fazer outra chamada à API de Chat Completions com a resposta da sua função para usar essa informação para criar uma resposta para o utilizador. 


![Fluxo de uma Chamada de Função](../../../../translated_images/pt-PT/LLM-Flow.3285ed8caf4796d7.webp)


### Elementos de uma chamada de função 

#### Entrada dos Utilizadores 

O primeiro passo é criar uma mensagem do utilizador. Esta pode ser atribuída dinamicamente ao tomar o valor de uma entrada de texto ou podes atribuir um valor aqui. Se esta é a tua primeira vez a trabalhar com a API de Completações de Chat, precisamos de definir o `role` e o `content` da mensagem. 

O `role` pode ser `system` (criando regras), `assistant` (o modelo) ou `user` (o utilizador final). Para a chamada de função, vamos atribuir isto como `user` e uma pergunta de exemplo. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Criar funções.

De seguida, vamos definir uma função e os parâmetros dessa função. Vamos usar apenas uma função aqui chamada `search_courses`, mas pode criar várias funções.

**Importante**: As funções são incluídas na mensagem do sistema para o LLM e serão incluídas na quantidade de tokens disponíveis que tem à sua disposição.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definições** 

`name` - O nome da função que queremos que seja chamada. 

`description` - Esta é a descrição de como a função funciona. Aqui é importante ser específico e claro 

`parameters` - Uma lista de valores e formato que pretende que o modelo produza na sua resposta 


`type` - O tipo de dados onde as propriedades serão guardadas 

`properties` - Lista dos valores específicos que o modelo irá usar para a sua resposta 


`name` - o nome da propriedade que o modelo irá usar na sua resposta formatada 

`type` - O tipo de dados desta propriedade 

`description` - Descrição da propriedade específica 


**Opcional**

`required` - propriedade obrigatória para que a chamada da função seja concluída 


### Fazer a chamada da função 
Após definir uma função, agora precisamos incluí-la na chamada à API de Chat Completion. Fazemo-lo adicionando `functions` ao pedido. Neste caso `functions=functions`. 

Existe também a opção de definir `function_call` para `auto`. Isto significa que deixamos o LLM decidir qual função deve ser chamada com base na mensagem do utilizador, em vez de a atribuirmos nós próprios.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Agora vamos analisar a resposta e ver como está formatada: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Pode ver que o nome da função é chamado e, a partir da mensagem do utilizador, o LLM conseguiu encontrar os dados para preencher os argumentos da função. 


## 3.Integração de Chamadas de Função numa Aplicação. 


Depois de termos testado a resposta formatada do LLM, agora podemos integrá-la numa aplicação. 

### Gerir o fluxo 

Para integrar isto na nossa aplicação, vamos seguir os seguintes passos: 

Primeiro, vamos fazer a chamada aos serviços Open AI e armazenar a mensagem numa variável chamada `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Agora vamos definir a função que irá chamar a API da Microsoft Learn para obter uma lista de cursos:


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Como melhor prática, iremos então verificar se o modelo pretende chamar uma função. Depois disso, vamos criar uma das funções disponíveis e correspondê-la à função que está a ser chamada. 
Depois, iremos apanhar os argumentos da função e mapeá-los para os argumentos do LLM.

Por fim, iremos acrescentar a mensagem de chamada da função e os valores que foram retornados pela mensagem `search_courses`. Isto fornece ao LLM toda a informação necessária para
responder ao utilizador utilizando linguagem natural. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Agora vamos enviar a mensagem atualizada para o LLM para que possamos receber uma resposta em linguagem natural em vez de uma resposta formatada em JSON para API. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Desafio de Código 

Excelente trabalho! Para continuar a sua aprendizagem sobre Azure Open AI Function Calling pode construir: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Mais parâmetros da função que podem ajudar os aprendizes a encontrar mais cursos. Pode encontrar os parâmetros de API disponíveis aqui: 
 - Criar outra chamada de função que recolha mais informações do aprendiz, como a sua língua nativa 
 - Criar tratamento de erros quando a chamada de função e/ou de API não devolver cursos adequados 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
